Unscrambling the files that contain the names of each gfx addon.

In [ ]:
from itertools import batched
from pathlib import Path
from PIL import Image
from IPython.display import display

In [ ]:
BASEDIR = Path("./tilesets/")

In [ ]:
for f in sorted(BASEDIR.glob("_jetp_?.dat", case_sensitive=False), key=lambda f: f.name.lower()):
    data = f.read_bytes()
    newdata = bytes(b ^ 52 for b in data)
    print("{:>12} {}".format(f.name, newdata.decode('ascii')))

In [ ]:
bin(52)

In [ ]:
hex(52)

---

Now exploring something else

In [ ]:
for f in BASEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    data = f.read_bytes()
    newdata = bytes(b ^ 48 for b in data)
    print(newdata[:48])

---

ImHex hexpat for `_JETP_?0.DAT` files:

---

In [ ]:
bodybytes = 3880

In [ ]:
bodynibbles = bodybytes * 2

In [ ]:
bodynibbles / 7

In [ ]:
320 * 200 / 970

In [ ]:
320*200

In [ ]:
320*200 / 1939

In [ ]:
1938*33 - 320*200

All green 66
FD9BF 66  (969 times)
FD9AB 66  (the last one)

All green 67
FD9FF 67
FD9EB 67

In [ ]:
'{:04b}'.format(0xB)

In [ ]:
'{:08b}'.format(0x67)

In [ ]:
'{:020b}'.format(1032255)

In [ ]:
'{:020b}'.format(1032319)

In [ ]:
'{:020b}'.format(1032383)

In [ ]:
'{:020b}'.format(1048511)

In [ ]:
'{:06b}'.format(13)

In [ ]:
'{:06b}'.format(44)

In [ ]:
'{:08b}'.format(0x68)

---

In [ ]:
class BitGenerator:
    def __init__(self, data: bytes):
        self.data = data
        self.pointer = 0
        self.buffer = []

    def remaining_bits(self):
        return len(self.buffer) + len(self.data) - self.pointer

    def is_empty(self):
        return self.pointer >= len(self.data) and len(self.buffer) == 0

    def consume_byte(self):
        out = self.data[self.pointer]
        self.pointer += 1
        return out

    def get_bit(self):
        if len(self.buffer) == 0:
            self.buffer.extend(int(bit) for bit in '{:08b}'.format(self.consume_byte()))
        return self.buffer.pop(0)

    def get_bits(self, how_many : int):
        out = 0
        for i in range(how_many):
            out <<= 1
            out |= self.get_bit()
        return out

    def get_byte(self):
        return self.get_bits(8)

In [ ]:
def gfxdat_parser(rawdata: bytes):
    # Raw palette has one byte per channel (R, G, B).
    # Each channel is 6-bit (0..63), as per VGA palette limitation.
    rawpalette = rawdata[:256*3]
    # Compressed stream of pixels.
    rawpixels = rawdata[256*3:]

    # One byte per channel (R, G, B), 8-bit per channel.
    palette = bytearray(round(c * 255 / 63) for c in rawpalette)

    # Our framebuffer where the compressed image will be uncompresed.
    pixels = bytearray(320*200)
    pointer = 0

    stream = BitGenerator(rawpixels)
    while pointer < len(pixels):
        is_repeating = stream.get_bit()
        qty = 1
        if is_repeating:
            qty = 2 + stream.get_bits(5)
        colorindex = stream.get_byte()
        for i in range(qty):
            pixels[pointer] = colorindex
            pointer += 1

    if (remainder := stream.remaining_bits()) >= 8:
        print("Warning: {} trailing bytes ({} bits) are being ignored after the pixel data.".format(remainder // 8, remainder))

    img = Image.new(mode="P", size=(320, 200))
    img.putpalette([c << 2 for c in rawpalette])
    img.putdata(pixels)

    return img

In [ ]:
for f in BASEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    display(gfxdat_parser(f.read_bytes()))

In [ ]:
default_palette = gfxdat_parser((BASEDIR / '_JETP_A0.DAT').read_bytes()).getpalette()

In [ ]:
def hexpat_color_from_palette(palette):
    return '\n'.join([
        '    match(p.color.byte) {',
        *(
            '        ({:>3}): return "{:02X}{:02X}{:02X}";'.format(i, r, g, b)
            for i, (r, g, b) in enumerate(batched(palette, 3))
        ),
        '    }',
    ])

In [ ]:
print(hexpat_color_from_palette(default_palette))